# Municipality to electricity price areas
Validate Sime municipality-price-area against Kartverket's municipality code list.

In [1]:
from pathlib import Path
import sys
from pyspark.sql import SparkSession, functions as F

from silver_layer.municipality_price_areas import (
    ingest,
    source_paths,
    municipality_codes,
)


PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
SILVER_ROOT = PROJECT_ROOT / "data" / "silver"

spark = (
    SparkSession.builder.appName("Municipality Price Areas Silver")
    .master("local[4]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

26/11/20 10:27:11 WARN Utils: Your hostname, homelab resolves to a loopback address: 127.0.1.1; using 192.168.10.53 instead (on interface lan0)
26/11/20 10:27:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 10:27:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/20 10:27:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/20 10:27:12 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## Ingest

In [2]:
result = ingest(
    bronze_root=BRONZE_ROOT,
    silver_root=SILVER_ROOT,
    start="2024-01-01",
    end="2026-01-01",
    spark=spark,
)
result

{
  "dataset": "municipality_price_areas",
  "status": "success",
  "rows": 368,
  "quarantined_rows": 0,
  "silver_path": "/home/roland/UiS/rewrite/group14/data/silver/geography/municipality_price_areas",
  "quarantine_path": "/home/roland/UiS/rewrite/group14/data/silver/quarantine/municipality_price_areas"
}


{'dataset': 'municipality_price_areas',
 'status': 'success',
 'rows': 368,
 'quarantined_rows': 0,
 'silver_path': '/home/roland/UiS/rewrite/group14/data/silver/geography/municipality_price_areas',
 'quarantine_path': '/home/roland/UiS/rewrite/group14/data/silver/quarantine/municipality_price_areas'}

In [ ]:
mapping = spark.read.parquet(result["silver_path"])
quarantine = spark.read.parquet(result["quarantine_path"])

print("Mapping relationships:", f"{mapping.count():,}")
print(
    "Unique municipalities:",
    f"{mapping.select('municipality_code').distinct().count():,}",
)
print("Quarantined Sime records:", f"{quarantine.count():,}")

mapping.printSchema()
mapping.orderBy("municipality_code", "price_area").show(15, truncate=False)

Mapping relationships: 368
Unique municipalities: 357
Quarantined Sime records: 0
root
 |-- municipality_code: string (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- price_area: string (nullable = true)
 |-- primary_price_area: string (nullable = true)
 |-- is_overlapping: boolean (nullable = true)

+-----------------+-----------------+----------+------------------+--------------+
|municipality_code|municipality_name|price_area|primary_price_area|is_overlapping|
+-----------------+-----------------+----------+------------------+--------------+
|0301             |Oslo             |NO1       |NO1               |false         |
|1101             |Eigersund        |NO2       |NO2               |false         |
|1103             |Stavanger        |NO2       |NO2               |false         |
|1106             |Haugesund        |NO2       |NO2               |false         |
|1108             |Sandnes          |NO2       |NO2               |false         |
|1111      

## 2. Data quality checks

The known Bronze contains 357 municipalities.

In [ ]:
sime_path, kartverket_path = source_paths(BRONZE_ROOT)
kartverket = municipality_codes(kartverket_path)
silver_codes = {
    row[0] for row in mapping.select("municipality_code").distinct().collect()
}

print("Kartverket municipalities:", len(kartverket))
print("Sime municipalities in Silver:", len(silver_codes))
print("Kartverket codes missing from Silver:", sorted(kartverket - silver_codes))
print("Silver codes absent from Kartverket:", sorted(silver_codes - kartverket))
print(
    "Duplicate (municipality, area) keys:",
    mapping.groupBy("municipality_code", "price_area")
    .count()
    .where("count > 1")
    .count(),
)
print(
    "Null required keys:",
    mapping.where("municipality_code IS NULL OR price_area IS NULL").count(),
)

Kartverket municipalities: 357
Sime municipalities in Silver: 357
Kartverket codes missing from Silver: []
Silver codes absent from Kartverket: []
Duplicate (municipality, area) keys: 0
Null required keys: 0


In [ ]:
print("Quarantine reasons:")
quarantine.groupBy("quarantine_reason").count().orderBy(F.desc("count")).show(
    truncate=False
)

print("Quarantine examples:")
quarantine.show(15, truncate=False)

Quarantine reasons:
+-----------------+-----+
|quarantine_reason|count|
+-----------------+-----+
+-----------------+-----+

Quarantine examples:
+-----------------+-----------------+------------------+-----------+-----------------+------------+
|municipality_code|municipality_name|primary_price_area|price_areas|quarantine_reason|_source_file|
+-----------------+-----------------+------------------+-----------+-----------------+------------+
+-----------------+-----------------+------------------+-----------+-----------------+------------+



## Overlapping municipalities

In [ ]:
overlaps = (
    mapping.where(F.col("is_overlapping"))
    .groupBy("municipality_code", "municipality_name", "primary_price_area")
    .agg(F.sort_array(F.collect_set("price_area")).alias("price_areas"))
    .orderBy("municipality_code")
)

print("Overlapping municipalities:", overlaps.count())
overlaps.show(30, truncate=False)

print("Price area counts:")
mapping.groupBy("price_area").count().orderBy("price_area").show()

Overlapping municipalities: 11
+-----------------+-----------------+------------------+-----------+
|municipality_code|municipality_name|primary_price_area|price_areas|
+-----------------+-----------------+------------------+-----------+
|3301             |Drammen          |NO1               |[NO1, NO2] |
|3303             |Kongsberg        |NO1               |[NO1, NO2] |
|3427             |Tynset           |NO1               |[NO1, NO3] |
|3429             |Folldal          |NO1               |[NO1, NO3] |
|3437             |Sel              |NO3               |[NO1, NO3] |
|4005             |Notodden         |NO2               |[NO1, NO2] |
|4618             |Ullensvang       |NO2               |[NO2, NO5] |
|4622             |Kvam             |NO5               |[NO2, NO5] |
|4624             |Bjørnafjorden    |NO5               |[NO2, NO5] |
|4638             |Høyanger         |NO3               |[NO3, NO5] |
|5045             |Grong            |NO3               |[NO3, NO4] |
+--